In [1]:
import h5py
import numpy as np
import pickle as pkl

In [5]:
with h5py.File("/home/mila/a/aidan.sirbu/track-mjx/data.h5", "r") as f:
    print("Keys: ", list(f.keys()))

Keys:  ['config', 'kp_data', 'kp_names', 'marker_sites', 'names_qpos', 'names_xpos', 'offsets', 'qpos', 'qvel', 'snips_order', 'xpos', 'xquat']


In [ ]:
with h5py.File("/home/mila/a/aidan.sirbu/track-mjx/data.h5", "r") as f:
    data = np.array(f["snips_order"])
    print(data.shape)

(842,)


In [14]:
with h5py.File("/home/mila/a/aidan.sirbu/track-mjx/data_fly.h5", "r") as f:
    print(f['all_clips']['joints'])
    print(f['all_clips']['position'].shape)

<HDF5 dataset "joints": shape (1730, 600, 36), type "<f4">
(1730, 600, 3)


In [25]:
# Find walking clips
def find_indices_with_walk(byte_array):
    """
    Takes a list of bytes objects, decodes them to strings, and returns the indices
    of any string that contains 'walk', case-insensitive.
    """
    indices = []
    for i, b in enumerate(byte_array):
        try:
            s = b.decode('utf-8')  # decode bytes to string
            if 'walk' in s.lower():  # case-insensitive search
                indices.append(i)
        except UnicodeDecodeError:
            continue  # skip bytes that can't be decoded
    return indices

In [28]:
print(find_indices_with_walk(data))

[2, 8, 15, 19, 22, 25, 30, 32, 33, 34, 35, 38, 41, 42, 43, 49, 51, 52, 54, 56, 58, 61, 67, 70, 73, 75, 76, 82, 83, 96, 97, 98, 102, 104, 105, 107, 109, 110, 111, 112, 116, 117, 122, 123, 124, 125, 129, 131, 139, 140, 143, 144, 145, 148, 149, 150, 152, 155, 156, 162, 163, 167, 169, 170, 171, 174, 176, 178, 179, 182, 184, 188, 189, 190, 191, 196, 197, 198, 201, 206, 209, 211, 212, 213, 215, 216, 219, 220, 224, 226, 230, 231, 232, 235, 238, 242, 250, 252, 253, 256, 257, 258, 263, 265, 266, 270, 272, 273, 277, 280, 281, 284, 287, 289, 293, 294, 295, 296, 297, 299, 300, 302, 305, 306, 307, 308, 309, 313, 315, 319, 326, 331, 333, 334, 335, 336, 337, 338, 344, 347, 348, 352, 353, 354, 358, 363, 364, 366, 367, 369, 370, 371, 372, 373, 374, 377, 378, 380, 381, 382, 386, 388, 389, 392, 393, 394, 395, 398, 399, 401, 403, 406, 408, 412, 416, 418, 419, 421, 426, 427, 432, 434, 436, 437, 438, 441, 442, 443, 445, 446, 447, 448, 451, 452, 453, 455, 458, 459, 463, 464, 470, 471, 472, 473, 475, 476, 478

In [15]:
def print_h5_structure(file_path, max_depth=None, show_data_preview=False):
    """
    Print the structure of an HDF5 file.
    
    Args:
        file_path: Path to the HDF5 file
        max_depth: Maximum depth to traverse (None for unlimited)
        show_data_preview: Whether to show small data previews
    """
    
    def print_item(name, obj, indent=0):
        """Recursively print structure of HDF5 objects"""
        if max_depth is not None and indent > max_depth:
            return
            
        spaces = "  " * indent
        
        if isinstance(obj, h5py.Dataset):
            print(f"{spaces}📄 Dataset: {name}")
            print(f"{spaces}   Shape: {obj.shape}")
            print(f"{spaces}   Dtype: {obj.dtype}")
            print(f"{spaces}   Size: {obj.size} elements")
            
            # Show data preview for small datasets
            if show_data_preview and obj.size < 50 and len(obj.shape) <= 2:
                print(f"{spaces}   Data: {obj[...]}")
            
            # Show attributes
            if obj.attrs:
                print(f"{spaces}   Attributes:")
                for attr_name, attr_value in obj.attrs.items():
                    print(f"{spaces}     {attr_name}: {attr_value}")
                    
        elif isinstance(obj, h5py.Group):
            print(f"{spaces}📁 Group: {name}")
            print(f"{spaces}   Contains {len(obj)} items")
            
            # Show attributes
            if obj.attrs:
                print(f"{spaces}   Attributes:")
                for attr_name, attr_value in obj.attrs.items():
                    print(f"{spaces}     {attr_name}: {attr_value}")
    
    print(f"📚 HDF5 File Structure: {file_path}")
    print("=" * 60)
    
    try:
        with h5py.File(file_path, 'r') as f:
            # Print file-level attributes
            if f.attrs:
                print("🏷️  File Attributes:")
                for attr_name, attr_value in f.attrs.items():
                    print(f"   {attr_name}: {attr_value}")
                print()
            
            # Print structure
            f.visititems(print_item)
            
            # Print summary
            print("\n" + "=" * 60)
            print("📊 Summary:")
            
            # Count items
            num_datasets = 0
            num_groups = 0
            
            def count_items(name, obj):
                nonlocal num_datasets, num_groups
                if isinstance(obj, h5py.Dataset):
                    num_datasets += 1
                elif isinstance(obj, h5py.Group):
                    num_groups += 1
            
            f.visititems(count_items)
            
            print(f"   Total groups: {num_groups}")
            print(f"   Total datasets: {num_datasets}")
            
    except Exception as e:
        print(f"❌ Error reading file: {e}")

# Test the function on your data file
print_h5_structure("/home/mila/a/aidan.sirbu/track-mjx/data_fly.h5")

📚 HDF5 File Structure: /home/mila/a/aidan.sirbu/track-mjx/data_fly.h5
📁 Group: all_clips
   Contains 8 items
📄 Dataset: all_clips/angular_velocity
   Shape: (1730, 600, 3)
   Dtype: float32
   Size: 3114000 elements
📄 Dataset: all_clips/body_positions
   Shape: (1730, 600, 68, 3)
   Dtype: float32
   Size: 211752000 elements
📄 Dataset: all_clips/body_quaternions
   Shape: (1730, 600, 68, 4)
   Dtype: float32
   Size: 282336000 elements
📄 Dataset: all_clips/joints
   Shape: (1730, 600, 36)
   Dtype: float32
   Size: 37368000 elements
📄 Dataset: all_clips/joints_velocity
   Shape: (1730, 600, 36)
   Dtype: float32
   Size: 37368000 elements
📄 Dataset: all_clips/position
   Shape: (1730, 600, 3)
   Dtype: float32
   Size: 3114000 elements
📄 Dataset: all_clips/quaternion
   Shape: (1730, 600, 4)
   Dtype: float32
   Size: 4152000 elements
📄 Dataset: all_clips/velocity
   Shape: (1730, 600, 3)
   Dtype: float32
   Size: 3114000 elements

📊 Summary:
   Total groups: 1
   Total datasets: 8


In [29]:
np.set_printoptions(threshold=np.inf)
with h5py.File("/home/mila/a/aidan.sirbu/track-mjx/data.h5", 'r') as f:
    mocap = f["names_qpos"][:]
    for idx, element in enumerate(mocap):
        print(f"Element {idx}: {element}")

Element 0: b'root'
Element 1: b'root'
Element 2: b'root'
Element 3: b'root'
Element 4: b'root'
Element 5: b'root'
Element 6: b'root'
Element 7: b'vertebra_1_extend'
Element 8: b'vertebra_2_bend'
Element 9: b'vertebra_3_twist'
Element 10: b'vertebra_4_extend'
Element 11: b'vertebra_5_bend'
Element 12: b'vertebra_6_twist'
Element 13: b'hip_L_supinate'
Element 14: b'hip_L_abduct'
Element 15: b'hip_L_extend'
Element 16: b'knee_L'
Element 17: b'ankle_L'
Element 18: b'toe_L'
Element 19: b'hip_R_supinate'
Element 20: b'hip_R_abduct'
Element 21: b'hip_R_extend'
Element 22: b'knee_R'
Element 23: b'ankle_R'
Element 24: b'toe_R'
Element 25: b'vertebra_C1_extend'
Element 26: b'vertebra_C1_bend'
Element 27: b'vertebra_C2_extend'
Element 28: b'vertebra_C2_bend'
Element 29: b'vertebra_C3_extend'
Element 30: b'vertebra_C3_bend'
Element 31: b'vertebra_C4_extend'
Element 32: b'vertebra_C4_bend'
Element 33: b'vertebra_C5_extend'
Element 34: b'vertebra_C5_bend'
Element 35: b'vertebra_C6_extend'
Element 36

In [25]:
# Script to find features (2nd dimension) that are zero for ALL time steps (1st dimension)
with h5py.File("/home/mila/a/aidan.sirbu/track-mjx/data.h5", 'r') as f:
    mocap = f["qpos"][:]
    
    print(f"Mocap shape: {mocap.shape}")
    print(f"Time steps: {mocap.shape[0]}, Features: {mocap.shape[1]}")
    print("=" * 50)
    
    # Find features that are zero across ALL time steps
    # Check which features (columns) have all zeros
    zero_features = []
    
    for feature_idx in range(mocap.shape[1]):
        # Check if this feature is zero for all time steps
        if np.all(mocap[:, feature_idx] == 0.0):
            zero_features.append(feature_idx)
    
    print(f"Features that are zero for ALL time steps: {zero_features}")

Mocap shape: (210500, 74)
Time steps: 210500, Features: 74
Features that are zero for ALL time steps: [18, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 57, 65, 73]


In [24]:
mocap[:, 66]

array([-8.04837886e-03, -1.08210221e-02, -1.74532924e-02, -1.74532924e-02,
       -1.74532924e-02, -1.74532924e-02, -1.74532924e-02, -1.74532924e-02,
       -1.74532924e-02, -1.74532924e-02, -1.74532924e-02, -1.74532924e-02,
       -1.74532924e-02, -1.74532924e-02, -1.74532924e-02, -1.74532924e-02,
       -1.74532924e-02, -1.74532924e-02, -1.59105100e-02, -1.70178451e-02,
        1.96627388e-03,  3.50891836e-02, -8.57620873e-03, -1.74532924e-02,
       -7.97095802e-03, -9.08561796e-03, -1.37284873e-02, -1.74532924e-02,
       -1.74532924e-02, -1.74532924e-02, -1.74532924e-02, -1.74532924e-02,
       -4.73734690e-03,  1.64412912e-02,  2.74558961e-02,  3.32545601e-02,
        3.75073478e-02,  3.70173529e-02,  3.30329835e-02,  2.84649450e-02,
        2.12017559e-02,  1.47592938e-02,  9.81065538e-03,  3.99668422e-03,
        3.13679548e-03,  9.40185878e-03,  1.45013360e-02,  2.33749449e-02,
        3.22610214e-02,  3.70477028e-02,  3.79546359e-02,  3.56020890e-02,
        3.14434767e-02,  